# Fallback: train YOLO26 in Colab with Ultralytics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/habibis007/wildfire-watch-yolo26/blob/main/notebooks/train_fallback_ultralytics.ipynb)

Use this only if Roboflow Train credits run short. It downloads the dataset version from
Roboflow, fine-tunes YOLO26 on a free T4, and uploads the weights back to the same Roboflow
version, so the demo notebook can still load the model by its ID.

Runtime → T4 GPU, and add `ROBOFLOW_API_KEY` as a Colab Secret.

In [ ]:
!pip install -q ultralytics roboflow
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = os.environ["ROBOFLOW_API_KEY"]

In [ ]:
WORKSPACE = "habeebs-workspace-gjewr"  # from the Roboflow URL
PROJECT   = "wildfire-watch-yolo26-cicrc"
VERSION   = 1
MODEL     = "yolo26n.pt"              # or yolo26s.pt
EPOCHS    = 60

from roboflow import Roboflow
version = Roboflow(api_key=API_KEY).workspace(WORKSPACE).project(PROJECT).version(VERSION)
# YOLO26 reads the same images/labels txt layout as the "yolov8" export.
ds = version.download("yolov8", location="/content/dataset")

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)  # COCO-pretrained starting point: this is the transfer-learning step
results = model.train(data=f"{ds.location}/data.yaml", epochs=EPOCHS, imgsz=640, batch=32,
                      patience=15, project="/content/runs", name="wildfire", exist_ok=True)

In [ ]:
from IPython.display import Image, display
run = "/content/runs/wildfire"
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    if os.path.exists(f"{run}/{f}"):
        display(Image(f"{run}/{f}", width=900))
metrics = YOLO(f"{run}/weights/best.pt").val(data=f"{ds.location}/data.yaml")
print(f"precision {metrics.box.mp:.3f}  recall {metrics.box.mr:.3f}  "
      f"mAP50 {metrics.box.map50:.3f}  mAP50-95 {metrics.box.map:.3f}")

In [ ]:
# Upload the weights to the same Roboflow version, so get_model("<project>/<version>") works.
version.deploy(model_type="yolo26", model_path=run, filename="weights/best.pt")